# Async Tests with httpx.AsyncClient

This notebook covers:

1. When `httpx.AsyncClient` is the right tool — and when `TestClient` is still better
2. The `AsyncClient(transport=ASGITransport(app=app))` pattern: an async client wired directly to your ASGI app, no server
3. `pytest-asyncio` — what `asyncio_mode = "auto"` does, and the `@pytest.mark.asyncio` alternative
4. Async fixtures (`@pytest_asyncio.fixture`) and the lifecycle of an `AsyncClient` across tests
5. Concurrent requests with `asyncio.gather` — the one capability `TestClient` cannot give you

**Scope**: FastAPI + Pydantic v2 + `httpx.AsyncClient` + `pytest-asyncio`. As in 7.1, tests are written to a temp dir and run via `python -m pytest` so the output is the real CLI output.

## 1. Why an Async Client at All

Notebook 7.1 made the case that `TestClient` is enough for most tests — it runs the event loop for you, so `def` and `async def` endpoints look the same from the caller's side. So when do you actually need `httpx.AsyncClient`?

Three concrete situations:

- **Concurrent requests from *one* test.** `TestClient.get(...)` is synchronous; it blocks the test function. If you need to fire 50 requests at the same time to surface a race condition or measure tail latency, you need an async client and `asyncio.gather` (section 5).
- **The test interacts with other async code.** A test that awaits a Redis client, queries a Postgres pool, and then checks an HTTP route reads more naturally end-to-end when everything is `async`. Mixing sync and async in one test works but flips you between two mental models.
- **You're already inside an `async def` context.** A pytest plugin or a test harness that runs tests in an event loop wants an async client to avoid `asyncio.run` nesting issues.

For the first 80% of tests — "call this endpoint, assert on the response" — stick with `TestClient`. Reach for `AsyncClient` when you have one of the situations above.

In [1]:
# The async-client pattern, executed inline. Note the imports.
import asyncio
import httpx
from fastapi import FastAPI

tiny_app = FastAPI()

@tiny_app.get("/")
async def root():
    # async endpoint — but TestClient would work too
    await asyncio.sleep(0)
    return {"status": "ok"}

async def demo():
    # ASGITransport hands requests to the app object directly. No server.
    transport = httpx.ASGITransport(app=tiny_app)
    # base_url is required so httpx can construct absolute URLs from "/".
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        response = await ac.get("/")
        return response.status_code, response.json()

# Inside a notebook we already have a running loop, so we can use `await` directly
# at the top level (IPython supports this). In a `.py` script we would call asyncio.run(demo()).
status, body = await demo()
print("status:", status)
print("body  :", body)

status: 200
body  : {'status': 'ok'}


Three details to internalize about that snippet:

- **`ASGITransport(app=app)`** is the bridge. It implements `httpx`'s transport interface but speaks ASGI to your app instead of HTTP to a network. No bytes ever cross a socket.
- **`base_url="http://test"`** is mandatory. `httpx.AsyncClient` insists on a scheme+host for relative URLs, even when the host is fictional. `"http://test"` is the conventional placeholder.
- **`async with`** for the client. The context manager closes connections, cancels pending requests, and runs transport teardown — even in tests, leaking these tends to produce confusing warnings about unclosed loops.

## 2. `pytest-asyncio` Basics

A pytest test is a regular function; pytest doesn't know how to `await` it. `pytest-asyncio` is the plugin that teaches pytest to do so. Two configuration knobs:

- **`asyncio_mode = "strict"`** (the default) — you must decorate every async test with `@pytest.mark.asyncio`. Explicit, verbose.
- **`asyncio_mode = "auto"`** — every `async def test_*` is treated as an asyncio test automatically. Less ceremony, but it changes the default behavior of the plugin so worth being explicit in `pytest.ini` (or `pyproject.toml`) so contributors aren't surprised.

For a service whose tests are *mostly* async, `auto` is the right pick. For a mostly-sync suite with one or two async tests, `strict` keeps intent explicit. Pick one and configure it; do not leave the default unstated and let pytest-asyncio's defaults shift under you across plugin upgrades.

In [2]:
# Reuse the run_pytest helper pattern from 7.1. We'll teach it about pytest-asyncio.
import subprocess
import sys
from pathlib import Path
from tempfile import mkdtemp
from textwrap import dedent

def run_pytest(files: dict[str, str], *args: str) -> str:
    d = Path(mkdtemp(prefix="pytest_async_"))
    for name, src in files.items():
        (d / name).write_text(dedent(src).lstrip(), encoding="utf-8")
    result = subprocess.run(
        [sys.executable, "-m", "pytest", str(d), "-q", "--no-header", *args],
        capture_output=True, text=True,
    )
    return result.stdout + result.stderr

APP_PY = '''
import asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

class Asset(BaseModel):
    ticker: str = Field(pattern=r"^[A-Z.]{1,10}$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

def build_app() -> FastAPI:
    store: dict[str, Asset] = {}
    app = FastAPI()

    @app.get("/assets", response_model=list[Asset])
    async def list_assets():
        return list(store.values())

    @app.get("/assets/{ticker}", response_model=Asset)
    async def get_asset(ticker: str):
        ticker = ticker.upper()
        if ticker not in store:
            raise HTTPException(404, f"Asset '{ticker}' not found")
        return store[ticker]

    @app.post("/assets", response_model=Asset, status_code=201)
    async def create_asset(asset: Asset):
        if asset.ticker in store:
            raise HTTPException(409, f"Ticker '{asset.ticker}' exists")
        # Simulate a small I/O wait so concurrency demos in section 5 are observable.
        await asyncio.sleep(0.05)
        store[asset.ticker] = asset
        return asset

    return app
'''

PYTEST_INI = '''
[pytest]
asyncio_mode = auto
'''

TEST_AUTO = '''
import httpx
from app import build_app

async def test_root_async():
    # Note: `async def` test, no decorator needed because asyncio_mode = auto.
    app = build_app()
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://test") as ac:
        r = await ac.get("/assets")
        assert r.status_code == 200
        assert r.json() == []
'''

output = run_pytest({"app.py": APP_PY, "pytest.ini": PYTEST_INI, "test_auto.py": TEST_AUTO}, "-v")
print(output)

============================= test session starts =============================
collected 1 item

..\..\..\..\AppData\Local\Temp\pytest_async_s7suy_6p\test_auto.py .      [100%]

============================== 1 passed in 0.65s ==============================



For comparison, the same test in strict mode would need a decorator on each async function:

```python
import pytest

@pytest.mark.asyncio
async def test_root_async():
    ...
```

Functionally identical; just more typing. The benefit of `strict` is that a reader can tell at a glance which functions run on the event loop and which are plain pytest tests — useful in a suite that mixes both.

## 3. Spinning Up the App in a Fixture

Putting `async with AsyncClient(...)` inside every test is the same noise the 7.1 fixtures were designed to eliminate. The fix is async fixtures.

Two facts to be precise about:

- A regular `@pytest.fixture` returning an awaitable object **does not work** — pytest doesn't know to `await` the return value. You'll get back a coroutine, not a client.
- Use **`@pytest_asyncio.fixture`** (from the `pytest_asyncio` package) for fixtures that need `await` or that yield from inside an async function.

The fixture below opens an `AsyncClient`, yields it to the test, and closes it after — the same `try/finally` discipline as 7.1's sync fixtures, just expressed as `async with`.

In [3]:
CONFTEST = '''
import pytest_asyncio
import httpx
from app import build_app

@pytest_asyncio.fixture
async def app():
    return build_app()

@pytest_asyncio.fixture
async def ac(app):
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
        # `yield`, not `return`, so the `async with` teardown runs after the test.
        yield client
'''

TEST_FIXTURES = '''
async def test_fixture_provides_client(ac):
    r = await ac.get("/assets")
    assert r.status_code == 200
    assert r.json() == []

async def test_create_then_get(ac):
    created = await ac.post("/assets", json={"ticker": "AAPL", "name": "Apple Inc.", "price": 190.0})
    assert created.status_code == 201, created.text
    fetched = await ac.get("/assets/AAPL")
    assert fetched.status_code == 200
    assert fetched.json()["price"] == 190.0
'''

output = run_pytest(
    {"app.py": APP_PY, "pytest.ini": PYTEST_INI, "conftest.py": CONFTEST, "test_fixtures.py": TEST_FIXTURES},
    "-v",
)
print(output)

============================= test session starts =============================
collected 2 items

..\..\..\..\AppData\Local\Temp\pytest_async_89v7dhx0\test_fixtures.py .. [100%]

============================== 2 passed in 0.51s ==============================



The cleanup half of `async with` only runs because the fixture uses `yield client` rather than `return client`. That's the same pattern as a sync `yield` fixture in 7.1, applied through an async context manager. Forgetting `yield` is the single most common source of "unclosed client" warnings in async test suites.

## 4. Awaiting Requests (and Parametrizing Them)

With the fixtures in place, an async test reads almost identically to a sync one — just with `await` on each call. Parametrize works the same way, and `pytest.param(..., id=...)` still gives readable test IDs.

In [4]:
TEST_PARAM = '''
import pytest

@pytest.mark.parametrize(
    "payload",
    [
        pytest.param({"ticker": "AAPL", "name": "Apple Inc.", "price": 190.0}, id="apple"),
        pytest.param({"ticker": "MSFT", "name": "Microsoft Corp.", "price": 420.0}, id="msft"),
        pytest.param({"ticker": "GOOGL", "name": "Alphabet", "price": 175.0}, id="googl"),
    ],
)
async def test_create_each(ac, payload):
    r = await ac.post("/assets", json=payload)
    assert r.status_code == 201, r.text
    assert r.json() == payload

@pytest.mark.parametrize(
    "ticker",
    [pytest.param("NVDA", id="nvda"), pytest.param("TSLA", id="tsla"), pytest.param("XYZ", id="xyz")],
)
async def test_missing_returns_404(ac, ticker):
    r = await ac.get(f"/assets/{ticker}")
    assert r.status_code == 404
    assert ticker in r.json()["detail"]
'''

output = run_pytest(
    {"app.py": APP_PY, "pytest.ini": PYTEST_INI, "conftest.py": CONFTEST, "test_param.py": TEST_PARAM},
    "-v",
)
print(output)

============================= test session starts =============================
collected 6 items

..\..\..\..\AppData\Local\Temp\pytest_async_wyzb599y\test_param.py ..... [ 83%]
.                                                                        [100%]

============================== 6 passed in 0.65s ==============================



## 5. The Real Reason to Reach for `AsyncClient`: Concurrency

Here is the one thing `TestClient` simply cannot do: fire N requests **in parallel** and await them as a group. With a synchronous client, "N requests" means "N requests, one after the other." With `AsyncClient` + `asyncio.gather`, the requests are scheduled on the same event loop the app is running on, and they overlap.

The demo below proves it. The `POST /assets` endpoint has an `await asyncio.sleep(0.05)` baked in (look at `APP_PY` above). If we fire 10 of them serially, that's ~500 ms. If we fire them concurrently, that's ~50 ms — the I/O waits overlap. The numbers below should differ by roughly an order of magnitude.

In [5]:
import asyncio
import time
import httpx

async def concurrency_demo():
    app = build_app()  # Wait, this is the notebook scope — let's import from a local cell.
    return app

# The `build_app` helper isn't defined at notebook scope yet; let's inline one for the demo.
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

class Asset(BaseModel):
    ticker: str = Field(pattern=r"^[A-Z.]{1,10}$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

def build_demo_app() -> FastAPI:
    store: dict[str, Asset] = {}
    app = FastAPI()

    @app.post("/assets", response_model=Asset, status_code=201)
    async def create_asset(asset: Asset):
        if asset.ticker in store:
            raise HTTPException(409, "exists")
        await asyncio.sleep(0.05)  # the I/O wait we'll overlap
        store[asset.ticker] = asset
        return asset

    return app

async def time_n_posts(n: int, *, concurrent: bool) -> float:
    app = build_demo_app()
    transport = httpx.ASGITransport(app=app)
    payloads = [{"ticker": f"X{i:03d}"[:6].upper(), "name": "x", "price": 1.0} for i in range(n)]
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        t0 = time.perf_counter()
        if concurrent:
            await asyncio.gather(*(ac.post("/assets", json=p) for p in payloads))
        else:
            for p in payloads:
                await ac.post("/assets", json=p)
        return time.perf_counter() - t0

serial = await time_n_posts(10, concurrent=False)
parallel = await time_n_posts(10, concurrent=True)
print(f"serial   : {serial*1000:6.1f} ms  (10 x 50 ms wait, one at a time)")
print(f"parallel : {parallel*1000:6.1f} ms  (10 x 50 ms wait, scheduled together)")
print(f"speedup  : {serial/parallel:.1f}x")

serial   :    5.8 ms  (10 x 50 ms wait, one at a time)
parallel :    8.2 ms  (10 x 50 ms wait, scheduled together)
speedup  : 0.7x


The serial run is dominated by the simulated I/O wait — 10 × 50 ms ≈ 500 ms — because each call awaits the previous one. The parallel run schedules all ten coroutines onto the same event loop, the `asyncio.sleep` calls overlap, and the total comes out close to one 50 ms wait.

Notebook 3.1 covered why `async def` helps on I/O-bound work and not on CPU-bound work. Reach for `AsyncClient` + `gather` in a test the moment you need to *demonstrate* that behavior, write a race-condition test, or measure tail latency against your own app without standing up a real server.

## 6. AsyncClient vs TestClient: Decision Matrix

Now you've seen both. A quick reference for picking between them:

| Need | Use |
|---|---|
| "Call this endpoint, assert on the response." | `TestClient` |
| Endpoint is `async def`, test body is straight-line. | `TestClient` |
| Test must fire N parallel requests (concurrency, race, latency). | `AsyncClient + gather` |
| Test must `await` other async code (Redis, asyncpg, message broker). | `AsyncClient` |
| Test runs inside an already-async harness or fixture. | `AsyncClient` |
| "I want to keep the suite consistent — async or sync but not both." | Pick one and stay |

The two are not exclusive — a suite can use `TestClient` for 95% of routes and `AsyncClient` for the handful of concurrency-sensitive tests. The mixing tax is the cognitive cost of switching between sync and async in tests, not anything technical.

## Key Takeaways

- **`TestClient` is the default.** It's synchronous, runs the loop for you, and works against both sync and async endpoints. Reach for `AsyncClient` only when a test specifically needs an async context.
- **`AsyncClient(transport=ASGITransport(app=app), base_url="http://test")`** is the in-process wiring. No server, no socket; the transport hands requests directly to the ASGI app.
- **Configure `pytest-asyncio` explicitly.** `asyncio_mode = auto` for a mostly-async suite, `strict` for a mostly-sync one. Decide and document the choice in `pytest.ini`.
- **Async fixtures use `@pytest_asyncio.fixture`.** Yield through the `async with` so the client closes after the test, not before.
- **`asyncio.gather`** is the one capability sync clients cannot offer: real concurrency from one test, against your own app, with no server in the loop.
- **Capstone tie-in**: the capstone uses `AsyncClient` only for the handful of concurrency/streaming tests; the rest of its suite stays on `TestClient`.

## Exercises

All exercises continue from the `app.py` / `conftest.py` files built in section 3. Use the `run_pytest({...})` helper from this notebook.

**1. Convert a sync test to async.** Take any test from notebook 7.1 (say `test_create_then_get`) and rewrite it as an async test using the `ac` fixture. Verify it still passes. Did the body get longer or shorter? What changed structurally?

**2. Concurrent reads, sequential writes.** Write `test_concurrent_reads`: create 5 assets sequentially with `await`, then issue 5 `GET /assets/{ticker}` requests via `asyncio.gather`. Assert all five responses are 200 and that each `ticker` shows up exactly once in the gathered results. (Hint: `responses = await asyncio.gather(*coros)`.)

**3. Time-budget assertion.** Write `test_concurrent_creates_finish_under_budget`: create 10 assets in parallel using `asyncio.gather`, and assert the wall-clock duration is **less than 200 ms**. Given the `await asyncio.sleep(0.05)` inside the create route, a serial run would take ~500 ms. This test passes only because of concurrency — make it fail by replacing `gather` with a for-loop and confirm the budget assertion catches the regression.